# Cell Setup

Python 3.10+; GPU opsional. Jalankan berurutan. Instalasi di bawah hanya komentar;
aktifkan manual bila perlu. Model diunduh saat eksekusi, bukan saat notebook dibuat.


In [ ]:
# %pip install "torch>=2.2,<3" "transformers>=4.44,<4.46" "numpy>=1.26,<3" "pandas>=2.2,<3" "scikit-learn>=1.5,<1.7" "nltk>=3.9.1,<4" "safetensors>=0.4,<1"
import os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
import gc
import hashlib
import importlib.metadata
import json
import math
import platform
import random
import re
import string
import uuid
from datetime import datetime, timezone
from pathlib import Path

import nltk
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import precision_recall_fscore_support
from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold
from torch.utils.data import DataLoader, Dataset
from transformers import AutoConfig, AutoModel, AutoTokenizer

# Tidak mengganti tokenizer diam-diam bila sumber NLTK tidak tersedia.
try:
    nltk.data.find("tokenizers/punkt_tab/english/")
except LookupError:
    if not nltk.download("punkt_tab", quiet=True):
        raise RuntimeError("Unduh NLTK punkt_tab gagal; sediakan resource sebelum melanjutkan.")


# Cell 1 — Konfigurasi

Baseline CodeBERT tanpa deduplikasi; split stratified independen per domain. Risiko leakage teks duplikat tetap dicatat.

Li et al. (2023): lima kelas, empat head, pelatihan task stokastik, CV 10-fold,
macro F1 kelas utang `[1,2,3,4]`; kelas 0 tetap dilatih. Table 4 **scratch** memakai
konvensi Kim: batch 50, Adadelta, loss tanpa bobot. Weighted loss dibahas kemudian
bersama FastText, bukan baseline scratch; rumus bobot tidak dirinci paper.

`EPOCHS=10`, `MAX_LEN=100`, seed 42 adalah pilihan implementasi lokal/warisan,
bukan fakta hyperparameter yang dilaporkan Li. Dropout 0.5 dan batas norma head 3
mengikuti referensi Kim. `lr=1.0` ditulis eksplisit sebagai default PyTorch Adadelta,
bukan nilai yang diklaim dilaporkan Li. Setting scratch ini **belum dituning untuk
CodeBERT**; AdamW `2e-5` hanya usulan eksperimen terpisah, bukan default di sini.
Tidak ada early stopping atau pemilihan model berdasarkan test fold.


In [ ]:
EXPERIMENT_NAME = 'codebert_satd'
DEDUPLICATE = False
PROJECT_ROOT = None  # Contoh Colab: Path("/content/drive/MyDrive/SATD-Research")
MODEL_NAME = "microsoft/codebert-base"
MODEL_REVISION = "main"  # Di-resolve sekali menjadi commit tetap sebelum semua fold.
N_SPLITS = 10
RUN_FOLDS = list(range(1, N_SPLITS + 1))  # Smoke: [1], bukan mengubah N_SPLITS.
EPOCHS = 10
SEED = 42
BATCH_SIZE = 50
MICRO_BATCH_SIZE = 5
GRADIENT_ACCUMULATION_STEPS = BATCH_SIZE // MICRO_BATCH_SIZE
LEARNING_RATE = 1.0
DROPOUT = 0.5
MAX_LEN = 100
MAX_NORM = 3.0
USE_CLASS_WEIGHTS = False  # Opsional: n/(5*n_c), dihitung hanya dari train fold.
GRAD_CLIP_NORM = None  # OFF; berbeda dari batas norma bobot classifier.
SAVE_CHECKPOINTS = False  # Menghemat penyimpanan; tidak memilih fold terbaik.
SOURCES = ["code_comments", "commit_messages", "pull_requests", "issues"]
LABELS = list(range(5))
DEBT_LABELS = [1, 2, 3, 4]
LABEL_MAP = {
    "non_debt": 0, "defect_debt": 0, "build_debt": 0, "architecture_debt": 0,
    "code_debt": 1, "design_debt": 1, "code/design_debt": 1,
    "requirement_debt": 2, "documentation_debt": 3, "test_debt": 4,
}
EXPECTED_COUNTS = {
    "code_comments": [58676, 2703, 757, 54, 85],
    "commit_messages": [4295, 522, 27, 98, 58],
    "pull_requests": [4301, 510, 20, 101, 68],
    "issues": [20089, 2169, 97, 487, 338],
}
assert BATCH_SIZE % MICRO_BATCH_SIZE == 0
assert EPOCHS >= 1 and N_SPLITS == 10
assert RUN_FOLDS and len(set(RUN_FOLDS)) == len(RUN_FOLDS)
assert set(RUN_FOLDS) <= set(range(1, N_SPLITS + 1))
FULL_CV = set(RUN_FOLDS) == set(range(1, N_SPLITS + 1))
print("FULL 10-FOLD CV" if FULL_CV else "PARTIAL: BUKAN hasil CV 10-fold lengkap")


# Cell 2 — Lokasi hasil dan seed

Hasil disimpan di direktori kerja atau `PROJECT_ROOT` jika diatur. Setiap run memakai folder baru.


In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True)


def save_json(path, payload):
    Path(path).write_text(json.dumps(payload, indent=2, ensure_ascii=False, allow_nan=False), encoding="utf-8")


root = Path(PROJECT_ROOT).expanduser().resolve() if PROJECT_ROOT is not None else Path.cwd().resolve()
run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S.%fZ") + "_" + uuid.uuid4().hex[:8]
run_dir = root / "Result" / EXPERIMENT_NAME / run_id
run_dir.mkdir(parents=True, exist_ok=False)
set_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Root:", root, "| Device:", device, "| Run:", run_dir)


# Cell 3 — Muat dan audit data

Empat CSV dibaca dari URL GitHub; hash dicatat dari data yang dibaca. `row_id` adalah indeks baris CSV asli.
Teks mentah dipertahankan; kosong tetap `""`. Label asing dan baris rusak menyebabkan
error, bukan dilewati. Hitungan kelas wajib cocok sebelum deduplikasi.


In [ ]:
from io import BytesIO
from urllib.request import urlopen

DATA_URLS = {'code_comments': 'https://github.com/NuriInformatics/retrain-multi-domain/raw/refs/heads/coba/data/satd-dataset-code_comments.csv', 'commit_messages': 'https://raw.githubusercontent.com/NuriInformatics/retrain-multi-domain/refs/heads/coba/data/satd-dataset-commit_messages.csv', 'issues': 'https://raw.githubusercontent.com/NuriInformatics/retrain-multi-domain/refs/heads/coba/data/satd-dataset-issues.csv', 'pull_requests': 'https://raw.githubusercontent.com/NuriInformatics/retrain-multi-domain/refs/heads/coba/data/satd-dataset-pull_requests.csv'}

frames, data_hashes = [], {}
for source in SOURCES:
    url = DATA_URLS[source]
    with urlopen(url) as response:
        csv_bytes = response.read()
    frame = pd.read_csv(BytesIO(csv_bytes), dtype=str, keep_default_na=False, on_bad_lines="error")
    missing = {"text", "classification", "project"} - set(frame.columns)
    if missing:
        raise ValueError(f"{source}: kolom wajib hilang: {sorted(missing)}")
    unknown = sorted(set(frame["classification"]) - set(LABEL_MAP))
    if unknown:
        raise ValueError(f"{source}: label tidak dikenal: {unknown}")
    frame = frame.rename(columns={"text": "raw_text"})
    frame["row_id"] = np.arange(len(frame), dtype=np.int64)
    frame["source"] = source
    frame["label"] = frame["classification"].map(LABEL_MAP).astype(int)
    observed = frame["label"].value_counts().reindex(LABELS, fill_value=0).tolist()
    assert observed == EXPECTED_COUNTS[source], (source, observed)
    frames.append(frame[["source", "row_id", "raw_text", "classification", "project", "label"]])
    data_hashes[url] = hashlib.sha256(csv_bytes).hexdigest()
original = pd.concat(frames, ignore_index=True)
original["normalized_text"] = original["raw_text"].str.replace(r"\s+", " ", regex=True).str.strip()
print(original.groupby(["source", "label"]).size().unstack(fill_value=0).reindex(SOURCES))


# Cell 4 — Kebijakan duplikat

Kunci adalah teks **mentah** dengan whitespace dirapatkan dan strip, case-sensitive,
independen dari label. Konflik label dicari global pada empat domain. Mode dedup
membuang **semua** instansi kunci konflik, lalu menyimpan baris pertama per
`(source, normalized_text)`; teks sama antardomain tetap boleh ada karena head berbeda.
Keputusan per baris dan audit disimpan; CSV masukan tidak diubah.


In [ ]:
conflict_keys = set(original.groupby("normalized_text", sort=False)["label"].nunique().loc[lambda s: s > 1].index)
original["has_label_conflict"] = original["normalized_text"].isin(conflict_keys)
original["decision"] = "retained"
if DEDUPLICATE:
    original.loc[original["has_label_conflict"], "decision"] = "excluded_label_conflict"
    eligible = original["decision"].eq("retained")
    duplicate_indices = original.loc[eligible].loc[
        lambda x: x.duplicated(["source", "normalized_text"], keep="first")
    ].index
    original.loc[duplicate_indices, "decision"] = "excluded_within_domain_duplicate"
data = original.loc[original["decision"].eq("retained")].copy().reset_index(drop=True)
assert not data.empty
if DEDUPLICATE:
    assert not data["has_label_conflict"].any()
    assert not data.duplicated(["source", "normalized_text"]).any()
audit_rows = []
for source in SOURCES:
    rows = original.loc[original["source"].eq(source)]
    audit_rows.append({
        "source": source, "original": len(rows),
        "observed_conflict_rows": int(rows["has_label_conflict"].sum()),
        "excluded_conflict": int(rows["decision"].eq("excluded_label_conflict").sum()),
        "excluded_duplicate": int(rows["decision"].eq("excluded_within_domain_duplicate").sum()),
        "final": int(rows["decision"].eq("retained").sum()),
    })
audit = pd.DataFrame(audit_rows)
audit.to_csv(run_dir / "data_audit.csv", index=False)
original.to_csv(run_dir / "row_decisions.csv", index=False)
print(audit.to_string(index=False))


# Cell 5 — Praproses kompatibel detector

Ikuti fungsi inferensi resmi: lowercase, hapus delimiter komentar, `sent_tokenize`,
TweetTokenizer, filter digit/karakter khusus, pertahankan `!` dan `?`.
Paper §3.4 juga menjelaskan kurasi bot, code block, dan karakter non-Inggris;
kurasi hulu CSV berlabel tidak diverifikasi ulang di notebook ini.
Setelah filter, `<pad>` dibuang karena padding CNN tidak relevan; teks bersih kosong
memakai special tokens CodeBERT saja. Pembersihan sama dengan baseline sebelum BPE.


In [ ]:
word_tokenizer = nltk.TweetTokenizer()
punctuation = string.punctuation.replace("!", "").replace("?", "")


def preprocess(raw_text):
    comment = re.sub(r"(//)|(/\*)|(\*/)", "", raw_text).lower()
    sentences = []
    for sentence in nltk.sent_tokenize(comment, language="english"):
        processed = []
        for token in word_tokenizer.tokenize(sentence):
            if token in {"non-nls", "<pad>"}:
                processed.append(token)
            elif token == ",":
                processed.append(".")
            elif any(char in token for char in (" ", ".", "#", "_", "/", ">", "<", "@")):
                continue
            elif any(char.isdigit() for char in token):
                continue
            else:
                processed.append(token)
        if processed and processed[-1] != ".":
            processed.append(".")
        sentences.extend(processed)
    stripped = [token for token in sentences if token and token not in punctuation
                and not any(char in token for char in (":", "=", ")", "("))]
    return " ".join(token for token in stripped if token != "<pad>")


data["clean_text"] = data["raw_text"].map(preprocess)
print("Teks bersih kosong:", int(data["clean_text"].eq("").sum()))


# Cell 6 — Manifest fold

Baseline: `StratifiedKFold` independen per domain, seed sama; duplikat dapat bocor
melalui encoder bersama. Overlap diamati, **tidak** diperbaiki diam-diam.
Dedup: satu `StratifiedGroupKFold` pada gabungan data, strata `source+label`, group
teks ternormalisasi; distribusi source-label hanya mendekati seimbang. Ini mengubah
protokol split juga, jadi **bukan efek dedup-only terkontrol**. Group bukan project;
generalisasi lintas project tidak diuji. Kunci bersih yang bertabrakan setelah praproses
masih mungkin ada; overlap tersebut juga dicatat, bukan diklaim hilang.


In [ ]:
data["fold"] = 0
if DEDUPLICATE:
    splitter = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
    strata = data["source"] + ":" + data["label"].astype(str)
    for fold, (_, test_idx) in enumerate(splitter.split(data, strata, groups=data["normalized_text"]), 1):
        assert data.loc[test_idx, "fold"].eq(0).all()
        data.loc[test_idx, "fold"] = fold
else:
    for source in SOURCES:
        indices = data.index[data["source"].eq(source)].to_numpy()
        splitter = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
        for fold, (_, local_test) in enumerate(splitter.split(indices, data.loc[indices, "label"]), 1):
            data.loc[indices[local_test], "fold"] = fold
assert data["fold"].between(1, N_SPLITS).all()
assert not data.duplicated(["source", "row_id"]).any()
overlap_rows = []
for fold in range(1, N_SPLITS + 1):
    train = data.loc[data["fold"].ne(fold)]
    test = data.loc[data["fold"].eq(fold)]
    overlap = set(train["normalized_text"]) & set(test["normalized_text"])
    clean_overlap = set(train["clean_text"]) & set(test["clean_text"])
    if DEDUPLICATE:
        assert not overlap, f"Fold {fold}: kebocoran group global"
    for source in SOURCES:
        train_labels = set(train.loc[train["source"].eq(source), "label"])
        assert train_labels == set(LABELS), (fold, source, train_labels)
        assert test["source"].eq(source).any(), (fold, source, "test kosong")
    overlap_rows.append({"fold": fold, "global_raw_key_overlap": len(overlap),
                         "global_clean_key_overlap": len(clean_overlap)})
data[["source", "row_id", "fold", "label", "normalized_text"]].to_csv(run_dir / "split_manifest.csv", index=False)
pd.DataFrame(overlap_rows).to_csv(run_dir / "split_overlap.csv", index=False)
print(pd.DataFrame(overlap_rows).to_string(index=False))
print(data.groupby(["fold", "source", "label"]).size().unstack(fill_value=0))


# Cell 7 — Tokenisasi dan revisi tetap

CodeBERT BPE diterapkan pada hasil praproses yang sama. Batas 100 **token BPE,
termasuk special tokens**, bukan 100 kata CNN. Cache token CPU dibuat sekali lalu
diindeks tiap fold. Resolusi `main` menghasilkan hash immutable yang dipakai untuk
config, tokenizer, dan semua encoder fold; bobot pretrained baru diunduh saat training.


In [ ]:
from huggingface_hub import HfApi

# Resolve revisi melalui API publik; jangan bergantung pada atribut privat config.
RESOLVED_REVISION = HfApi().model_info(MODEL_NAME, revision=MODEL_REVISION).sha
if not RESOLVED_REVISION or not re.fullmatch(r"[0-9a-f]{40}", RESOLVED_REVISION):
    raise RuntimeError("Commit model immutable tidak dapat di-resolve; jangan melanjutkan dengan revisi bergerak.")
resolved_config = AutoConfig.from_pretrained(
    MODEL_NAME, revision=RESOLVED_REVISION, trust_remote_code=False,
)
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME, revision=RESOLVED_REVISION, trust_remote_code=False, use_fast=True,
)
cache_parts = {}
for start in range(0, len(data), 1024):
    batch = tokenizer(data["clean_text"].iloc[start:start + 1024].tolist(),
                      padding="max_length", truncation=True, max_length=MAX_LEN,
                      return_tensors="pt")
    for key in tokenizer.model_input_names:
        if key in batch:
            cache_parts.setdefault(key, []).append(batch[key].cpu())
token_cache = {key: torch.cat(parts, dim=0) for key, parts in cache_parts.items()}
del cache_parts
label_cache = torch.tensor(data["label"].to_numpy(), dtype=torch.long)
assert all(values.shape == (len(data), MAX_LEN) for values in token_cache.values())
print("Commit:", RESOLVED_REVISION, "| Cache:", {key: tuple(value.shape) for key, value in token_cache.items()})


# Cell 8 — Metadata run

Simpan parameter, SHA256 masukan, revisi tokenizer/model, versi paket, dan protokol.
Run parsial ditandai eksplisit. Artefak fold bukan model produksi.


In [ ]:
# Konfigurasi inti dicatat agar hasil bisa ditelusuri.
settings = {
    "experiment_name": EXPERIMENT_NAME,
    "deduplicate": DEDUPLICATE,
    "paper_doi": "10.1007/s10664-023-10297-9",
    "source_commit": "df412d88c12c42164dc7574389070f7ef2cc1d12",
    "source_url": "https://github.com/yikun-li/satd-different-sources-data/blob/df412d88c12c42164dc7574389070f7ef2cc1d12/SATD%20Detector/satd_detector.py",
    "model_name": MODEL_NAME,
    "requested_revision": MODEL_REVISION,
    "resolved_revision": RESOLVED_REVISION,
    "tokenizer_revision": RESOLVED_REVISION,
    "tokenizer_class": type(tokenizer).__name__,
    "hidden_size": int(resolved_config.hidden_size),
    "n_splits": N_SPLITS, "run_folds": RUN_FOLDS, "full_cv": FULL_CV,
    "epochs": EPOCHS, "seed": SEED, "batch_size": BATCH_SIZE,
    "micro_batch_size": MICRO_BATCH_SIZE,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
    "optimizer": "Adadelta", "learning_rate": LEARNING_RATE, "rho": 0.9, "eps": 1e-6,
    "dropout": DROPOUT, "max_length_bpe_including_specials": MAX_LEN,
    "classifier_row_max_norm": MAX_NORM, "use_class_weights": USE_CLASS_WEIGHTS,
    "class_weight_formula_if_enabled": "n_train / (5 * n_class_train), no cap",
    "gradient_clip_norm": GRAD_CLIP_NORM, "save_checkpoints": SAVE_CHECKPOINTS,
    "sources": SOURCES, "label_map": LABEL_MAP, "metric_labels": DEBT_LABELS,
    "epoch_updates": "sum(ceil(n_train_domain / effective_batch_size))",
    "task_sampling": "uniform with replacement, restart shuffled loader when exhausted",
    "dedup_key": "raw text, collapse whitespace, strip, case-sensitive",
    "split": "pooled StratifiedGroupKFold source+label" if DEDUPLICATE else "per-domain StratifiedKFold",
    "preprocessing": "detector-compatible cleaning; omit <pad>; then CodeBERT BPE",
    "upstream_curation": "labeled CSV assumed curated; paper bot/code-block/non-English filtering not revalidated",
    "data_sha256": data_hashes,
    "reference_caveat": "scratch optimizer/batch/unweighted setting, not tuned for CodeBERT; epochs/length/seed local",
}
save_json(run_dir / "run_config.json", settings)

**Lingkungan eksekusi.** Versi paket dan perangkat disimpan terpisah dari konfigurasi eksperimen.

In [ ]:
packages = ["torch", "transformers", "numpy", "pandas", "scikit-learn", "nltk", "safetensors"]
versions = {name: importlib.metadata.version(name) for name in packages}
environment = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "python": platform.python_version(),
    "platform": platform.platform(),
    "packages": versions,
    "device": str(device),
    "cuda_version": torch.version.cuda,
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "torch_settings": {
        "deterministic_algorithms": torch.are_deterministic_algorithms_enabled(),
        "cudnn_deterministic": torch.backends.cudnn.deterministic,
        "cudnn_benchmark": torch.backends.cudnn.benchmark,
        "cudnn_allow_tf32": torch.backends.cudnn.allow_tf32,
        "matmul_allow_tf32": torch.backends.cuda.matmul.allow_tf32,
        "CUBLAS_WORKSPACE_CONFIG": os.environ.get("CUBLAS_WORKSPACE_CONFIG"),
    },
    "settings": settings,
}
save_json(run_dir / "environment.json", environment)

# Cell 9 — Dataset dan model

Satu encoder `microsoft/codebert-base` bersama, empat head linear lima kelas;
representasi token pertama, dropout 0.5. Hidden size dibaca dari config (umumnya 768).
Semua lapisan encoder di-fine-tune. CNN/embedding diganti, bukan reproduksi arsitektur asli.


In [ ]:
class CachedDataset(Dataset):
    def __init__(self, indices):
        self.indices = np.asarray(indices, dtype=np.int64)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, index):
        row = int(self.indices[index])
        features = {key: values[row] for key, values in token_cache.items()}
        return features, label_cache[row], row


class MultitaskCodeBERT(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(
            MODEL_NAME, revision=RESOLVED_REVISION, trust_remote_code=False,
        )
        self.dropout = nn.Dropout(DROPOUT)
        self.heads = nn.ModuleDict({source: nn.Linear(self.encoder.config.hidden_size, len(LABELS))
                                    for source in SOURCES})
        for parameter in self.encoder.parameters():
            parameter.requires_grad_(True)

    def forward(self, features, source):
        hidden = self.encoder(**features).last_hidden_state[:, 0, :]
        return self.heads[source](self.dropout(hidden))

    @torch.no_grad()
    def constrain_classifier(self):
        # Batas L2 per baris bobot head; bukan gradient clipping.
        for head in self.heads.values():
            norms = head.weight.norm(p=2, dim=1, keepdim=True).clamp_min(1e-12)
            head.weight.mul_((MAX_NORM / norms).clamp(max=1.0))


def train_weights(indices):
    counts = torch.bincount(label_cache[indices], minlength=len(LABELS)).float()
    assert bool((counts > 0).all()), "Kelas train hilang"
    if USE_CLASS_WEIGHTS:
        return (counts.sum() / (len(LABELS) * counts)).to(device)
    return None


# Cell 10 — Update efektif

Setiap update memilih domain uniform dengan replacement. Loader batch efektif 50
dipecah menjadi microbatch 5 (10 akumulasi saat penuh); batch sisa tetap benar.
Jika loader habis, buat iterator baru dengan shuffle baru, bukan `itertools.cycle`.
Denominator loss adalah jumlah bobot target **seluruh batch efektif** (atau jumlah
contoh tanpa bobot), bukan rata-rata mean microbatch. Dropout membuat realisasi
stokastik berbeda dari satu forward batch besar, tetapi normalisasi loss tetap benar.
Jumlah update per epoch = jumlah `ceil(n_train_domain/50)`; konvensi lokal ini tidak
ditentukan paper. Tidak ada AMP atau DataParallel.


In [ ]:
def effective_update(model, optimizer, features, targets, source, weights):
    optimizer.zero_grad(set_to_none=True)
    denominator = (weights[targets.to(device)].sum() if weights is not None
                   else torch.tensor(float(len(targets)), device=device))
    if not bool(torch.isfinite(denominator)) or denominator.item() <= 0:
        raise FloatingPointError("Denominator loss tidak valid")
    effective_loss = 0.0
    for start in range(0, len(targets), MICRO_BATCH_SIZE):
        stop = start + MICRO_BATCH_SIZE
        micro_features = {key: values[start:stop].to(device) for key, values in features.items()}
        micro_targets = targets[start:stop].to(device)
        logits = model(micro_features, source)
        loss = F.cross_entropy(logits, micro_targets, weight=weights, reduction="sum") / denominator
        if not bool(torch.isfinite(loss)):
            raise FloatingPointError(f"Loss non-finite: {source}")
        loss.backward()
        effective_loss += float(loss.detach())
    # Sekaligus periksa gradien; max_norm=inf tidak mengubah gradien bila clipping OFF.
    grad_norm = torch.nn.utils.clip_grad_norm_(
        model.parameters(), float("inf") if GRAD_CLIP_NORM is None else GRAD_CLIP_NORM,
        error_if_nonfinite=True,
    )
    optimizer.step()
    model.constrain_classifier()
    return effective_loss


def make_loaders(fold):
    train_loaders, test_loaders, weights = {}, {}, {}
    for source_index, source in enumerate(SOURCES):
        source_mask = data["source"].eq(source)
        train_idx = data.index[source_mask & data["fold"].ne(fold)].to_numpy()
        test_idx = data.index[source_mask & data["fold"].eq(fold)].to_numpy()
        generator = torch.Generator().manual_seed(SEED + fold * 100 + source_index)
        train_loaders[source] = DataLoader(CachedDataset(train_idx), batch_size=BATCH_SIZE,
                                            shuffle=True, drop_last=False, num_workers=0,
                                            generator=generator)
        test_loaders[source] = DataLoader(CachedDataset(test_idx), batch_size=MICRO_BATCH_SIZE,
                                           shuffle=False, drop_last=False, num_workers=0)
        weights[source] = train_weights(train_idx)
    return train_loaders, test_loaders, weights


# Cell 11 — Pelatihan tiap fold

Model dan optimizer baru tiap fold. Test hanya dievaluasi **setelah epoch terakhir**;
tidak memilih epoch/fold terbaik. Checkpoint final bersifat opsional dan selalu
bernomor fold, memuat state model, tokenizer, config encoder, dan metadata run.
CPU boleh tetapi fine-tuning 10-fold dapat sangat lambat; smoke `[1]` tetap memakai
split 10-fold, atur `EPOCHS` terpisah bila diperlukan.


In [ ]:
def train_one_fold(model, optimizer, train_loaders, weights, fold, training_rows):
    updates_per_epoch = sum(len(loader) for loader in train_loaders.values())
    task_rng = random.Random(SEED + fold)

    for epoch in range(1, EPOCHS + 1):
        model.train()
        iterators = {source: iter(loader) for source, loader in train_loaders.items()}
        losses = []
        task_updates = dict.fromkeys(SOURCES, 0)

        for _ in range(updates_per_epoch):
            source = task_rng.choice(SOURCES)
            try:
                features, targets, _ = next(iterators[source])
            except StopIteration:
                iterators[source] = iter(train_loaders[source])
                features, targets, _ = next(iterators[source])
            loss = effective_update(model, optimizer, features, targets, source, weights[source])
            losses.append(loss)
            task_updates[source] += 1

        mean_loss = float(np.mean(losses))
        training_rows.append({
            "fold": fold, "epoch": epoch, "mean_loss": mean_loss,
            "updates": updates_per_epoch,
            **{f"updates_{source}": count for source, count in task_updates.items()},
        })
        pd.DataFrame(training_rows).to_csv(run_dir / "training_trace.csv", index=False)
        print(f"Fold {fold}/{N_SPLITS} epoch {epoch}/{EPOCHS}: loss={mean_loss:.6f}; updates={task_updates}")

**Evaluasi fold.** Hitung prediksi dan F1 untuk setiap sumber setelah pelatihan selesai.

In [ ]:
def evaluate_one_fold(model, test_loaders, fold, prediction_rows, metric_rows):
    model.eval()
    domain_scores = []
    with torch.no_grad():
        for source in SOURCES:
            trues, preds = [], []
            for features, targets, row_indices in test_loaders[source]:
                features = {key: values.to(device) for key, values in features.items()}
                predicted = model(features, source).argmax(dim=1).cpu().tolist()
                truth = targets.tolist()
                trues.extend(truth)
                preds.extend(predicted)
                for row_index, actual, prediction in zip(row_indices.tolist(), truth, predicted):
                    prediction_rows.append({
                        "source": source, "row_id": int(data.at[row_index, "row_id"]),
                        "true": actual, "pred": prediction, "fold": fold,
                    })

            precision, recall, f1, support = precision_recall_fscore_support(
                trues, preds, labels=LABELS, zero_division=0,
            )
            debt_macro = float(np.mean(f1[DEBT_LABELS]))
            domain_scores.append(debt_macro)
            for label in LABELS:
                metric_rows.append({
                    "fold": fold, "source": source, "label": label,
                    "precision": float(precision[label]), "recall": float(recall[label]),
                    "f1": float(f1[label]), "support": int(support[label]),
                    "debt_macro_f1": debt_macro, "all_class_macro_f1": float(np.mean(f1)),
                })
            print(f"Fold {fold} {source}: debt-only macro F1={debt_macro:.6f}")
    return float(np.mean(domain_scores))

**Checkpoint opsional.** Simpan model epoch terakhir, bukan model yang dipilih berdasarkan test.

In [ ]:
def save_checkpoint(model, weights, fold):
    checkpoint_dir = run_dir / f"fold_{fold:02d}"
    checkpoint_dir.mkdir(exist_ok=False)
    torch.save({
        "model_state_dict": model.state_dict(),
        "fold": fold,
        "settings": settings,
        "class_weights": {
            source: weight.detach().cpu().tolist() if weight is not None else None
            for source, weight in weights.items()
        },
    }, checkpoint_dir / "checkpoint.pt")
    tokenizer.save_pretrained(checkpoint_dir / "tokenizer")
    model.encoder.config.save_pretrained(checkpoint_dir / "encoder_config")
    save_json(checkpoint_dir / "metadata.json", {
        "fold": fold, "settings": settings,
        "selection": "final epoch; not selected by test; not production",
    })

**Jalankan fold.** Inisialisasi model baru, latih, evaluasi, lalu simpan hasil.

In [ ]:
metric_rows, prediction_rows, fold_score_rows, training_rows = [], [], [], []
for fold in RUN_FOLDS:
    set_seed(SEED + fold)
    model = MultitaskCodeBERT().to(device)
    optimizer = torch.optim.Adadelta(model.parameters(), lr=LEARNING_RATE, rho=0.9, eps=1e-6)
    train_loaders, test_loaders, weights = make_loaders(fold)

    train_one_fold(model, optimizer, train_loaders, weights, fold, training_rows)
    overall = evaluate_one_fold(model, test_loaders, fold, prediction_rows, metric_rows)
    fold_score_rows.append({"fold": fold, "overall_debt_macro_f1": overall})
    print(f"Fold {fold}: overall={overall:.6f}")

    if SAVE_CHECKPOINTS:
        save_checkpoint(model, weights, fold)
    pd.DataFrame(metric_rows).to_csv(run_dir / "fold_metrics.csv", index=False)
    pd.DataFrame(prediction_rows).to_csv(run_dir / "predictions.csv", index=False)
    pd.DataFrame(fold_score_rows).to_csv(run_dir / "fold_scores.csv", index=False)
    del model, optimizer, train_loaders, test_loaders, weights
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# Cell 12 — Cakupan OOF

Setiap baris terpilih wajib diprediksi tepat sekali; full CV mencakup semua baris
retained. Run parsial hanya mencakup test fold yang diminta, bukan hasil CV lengkap.


In [ ]:
predictions = pd.DataFrame(prediction_rows)
fold_metrics = pd.DataFrame(metric_rows)
fold_scores = pd.DataFrame(fold_score_rows)
assert not predictions.duplicated(["source", "row_id"]).any()
expected = data.loc[data["fold"].isin(RUN_FOLDS), ["source", "row_id", "label", "fold"]]
coverage = expected.merge(predictions, on=["source", "row_id", "fold"], how="outer", validate="one_to_one", indicator=True)
assert coverage["_merge"].eq("both").all()
assert coverage["label"].eq(coverage["true"]).all()
assert len(predictions) == len(expected)
if FULL_CV:
    assert len(predictions) == len(data)
coverage_info = {"full_cv": FULL_CV, "completed_folds": RUN_FOLDS,
                 "oof_rows": len(predictions), "retained_rows": len(data),
                 "status": "complete_10_fold_cv" if FULL_CV else "partial_not_full_cv"}
save_json(run_dir / "coverage.json", coverage_info)
print(coverage_info)


# Cell 13 — Ringkasan numerik

Rata-rata aritmetika dan standar deviasi sampel (`ddof=1`) antar-fold, tanpa pembulatan
pada CSV. Overall adalah rata-rata empat domain per fold, bukan pembobotan jumlah
contoh. Run satu fold menghasilkan std kosong/NaN, bukan ketidakpastian nol.
Macro utama mengecualikan kelas 0; metrik tiap kelas termasuk 0 tetap tersedia.


In [ ]:
summary_rows = []


def add_summary(source, metric, values, label=None):
    values = np.asarray(values, dtype=float)
    summary_rows.append({"source": source, "metric": metric, "label": label,
                         "mean": float(np.mean(values)),
                         "std": float(np.std(values, ddof=1)) if len(values) > 1 else np.nan,
                         "n_folds": len(values), "full_cv": FULL_CV})


for source in SOURCES:
    source_metrics = fold_metrics.loc[fold_metrics["source"].eq(source)]
    for label in LABELS:
        label_metrics = source_metrics.loc[source_metrics["label"].eq(label)]
        for metric in ["precision", "recall", "f1", "support"]:
            add_summary(source, metric, label_metrics[metric], label)
    per_fold = source_metrics.drop_duplicates("fold")
    add_summary(source, "debt_macro_f1", per_fold["debt_macro_f1"])
    add_summary(source, "all_class_macro_f1", per_fold["all_class_macro_f1"])
add_summary("overall_domain_average", "debt_macro_f1", fold_scores["overall_debt_macro_f1"])
summary = pd.DataFrame(summary_rows)
summary.to_csv(run_dir / "summary.csv", index=False)
print("FULL CV" if FULL_CV else "PARTIAL: BUKAN FULL CV")
print(summary.loc[summary["metric"].eq("debt_macro_f1")].to_string(index=False))
print("Artefak:", run_dir)


# Cell 14 — Batas interpretasi

Table 4 scratch MT-Text-CNN `[0.619, 0.475, 0.441, 0.453]` (urutan komentar,
commit, pull request, issue) hanya referensi numerik jurnal, **bukan target CodeBERT,
bukan bukti reproduksi**. Encoder pretrained dan tokenisasi berbeda dari CNN;
setting scratch tidak menjamin konvergensi transformer. Deduplikasi plus grouped split
mengubah dua aspek protokol. Hash, audit, overlap, dan manifest perlu diperiksa sebelum
membandingkan hasil. Tidak ada inferensi atau pemilihan checkpoint otomatis.


In [ ]:
# Referensi opsional, tidak dipakai untuk training, seleksi model, atau klaim keberhasilan.
TABLE4_SCRATCH_REFERENCE = dict(zip(SOURCES, [0.619, 0.475, 0.441, 0.453]))
print("Referensi jurnal MT-Text-CNN scratch, bukan hasil CodeBERT:", TABLE4_SCRATCH_REFERENCE)
